PREPROCESAMIENTO

In [4]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS 
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import f1_score, classification_report


# --- 1. Carga de Datos y Definición de Etiquetas (Punto Crítico) ---
print("--- Paso 1: Intentando Cargar el DataFrame ---")
try:
    # Intenta cargar el archivo. Asume que está en la misma carpeta.
    df = pd.read_csv('../data/raw/youtoxic_english_1000.csv') 
    
    # Si la carga fue exitosa, imprimimos una verificación
    print("✅ DataFrame cargado exitosamente. Las primeras 5 filas son:")
    print(df.head())
    
    # Limpieza inicial y conversión de tipos
    df['Text'].fillna('', inplace=True)
    
    toxic_cols = ['IsToxic', 'IsAbusive', 'IsThreat', 'IsProvocative', 'IsObscene', 
                  'IsHatespeech', 'IsRacist', 'IsNationalist', 'IsSexist', 
                  'IsHomophobic', 'IsReligiousHate', 'IsRadicalism']
    for col in toxic_cols:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(bool).astype(int) 
        elif df[col].dtype != 'int64':
            df[col] = df[col].astype(int)
            
except FileNotFoundError:
    print("❌ ERROR CRÍTICO: El archivo 'youtoxic_english_1000.csv' NO fue encontrado.")
    print("Asegúrate de que el archivo está en la misma carpeta que el script de Python.")
    exit() # Detiene la ejecución si falla la carga.

# --- 2. Preprocesamiento Simplificado (Sin NLTK) ---

def preprocess_text_simplified(text):
    """Limpia y simplifica el texto para la vectorización."""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # URLs
    text = re.sub(r'@\w+|#\w+', '', text) # Menciones y Hashtags
    text = re.sub(r'[^\w\s]', '', text) # Puntuación y especiales
    text = re.sub(r'\b\d+\b', '', text) # Números
    text = re.sub(r'\s+', ' ', text).strip() # Espacios múltiples
    
    tokens = [word for word in text.split() if len(word) > 2]
    
    return " ".join(tokens)

print("\n--- Paso 2: Aplicando Preprocesamiento Simplificado ---")
df['Text_Processed'] = df['Text'].apply(preprocess_text_simplified)
print("Preprocesamiento completado. Muestra:", df['Text_Processed'].head().tolist())

# --- 3. Separación de Datos (Train/Test Split) ---

X = df['Text_Processed']
Y = df['IsToxic'] # Etiqueta principal para estratificación

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, 
    test_size=0.2, 
    random_state=42, 
    stratify=Y 
)

print(f"\n--- Paso 3: Separación de Datos ---")
print(f"Datos separados: {len(X_train)} muestras de entrenamiento, {len(X_test)} de prueba.")

# --- 4. Vectorización TF-IDF (N-Grama) ---

# Conversión de frozenset a list para evitar InvalidParameterError
stop_words_list = list(ENGLISH_STOP_WORDS) 

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), # Unigramas y Bigramas
    stop_words=stop_words_list,
    max_df=0.9, 
    min_df=5 
)

X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

print("\n--- Paso 4: Vectorización TF-IDF ---")
print(f"Dimensión de los datos de Entrenamiento: {X_train_vectorized.shape}")
print(f"Número total de características (N-gramas): {X_train_vectorized.shape[1]}")

# --- 5. Modelización: Entrenamiento y Evaluación ---

Y_multi_train = df.iloc[X_train.index][toxic_cols] 
Y_multi_test = df.iloc[X_test.index][toxic_cols]

base_clf = ComplementNB()
model_ovr = OneVsRestClassifier(base_clf)

print("\n--- Paso 5: Entrenando Modelo CNB Multi-Etiqueta ---")
model_ovr.fit(X_train_vectorized, Y_multi_train)
print("Entrenamiento completado.")

Y_pred = model_ovr.predict(X_test_vectorized)

f1_micro = f1_score(Y_multi_test, Y_pred, average='micro', zero_division=0)

print("\n--- Resultados de la Evaluación ---")
print(f"Micro F1-Score del Modelo (clave para desbalance): {f1_micro:.4f}")
print("\nInforme de Clasificación (Recuerda que estas son solo 200 muestras de prueba):")
print(classification_report(Y_multi_test, Y_pred, target_names=toxic_cols, zero_division=0))

--- Paso 1: Intentando Cargar el DataFrame ---
✅ DataFrame cargado exitosamente. Las primeras 5 filas son:
              CommentId      VideoId  \
0  Ugg2KwwX0V8-aXgCoAEC  04kJtp6pVXI   
1  Ugg2s5AzSPioEXgCoAEC  04kJtp6pVXI   
2  Ugg3dWTOxryFfHgCoAEC  04kJtp6pVXI   
3  Ugg7Gd006w1MPngCoAEC  04kJtp6pVXI   
4  Ugg8FfTbbNF8IngCoAEC  04kJtp6pVXI   

                                                Text  IsToxic  IsAbusive  \
0  If only people would just take a step back and...    False      False   
1  Law enforcement is not trained to shoot to app...     True       True   
2  \nDont you reckon them 'black lives matter' ba...     True       True   
3  There are a very large number of people who do...    False      False   
4  The Arab dude is absolutely right, he should h...    False      False   

   IsThreat  IsProvocative  IsObscene  IsHatespeech  IsRacist  IsNationalist  \
0     False          False      False         False     False          False   
1     False          False      Fal

/var/folders/gt/tfdp3g0n2430nwvd1ftk_4w80000gn/T/ipykernel_32568/3344038844.py:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Text'].fillna('', inplace=True)
/Users/aroamateogomez/Desktop/BootcampIA/Proyecto10/Proyecto_10_X/proyecto10-grupo5/.venv/lib/python3.11/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 9 is present in all training examples.
  warnings.warn(
/Users/aroamateogomez/Desktop/BootcampIA/Proyecto10/Proyecto_10_X/proyecto10-grupo5/.venv/lib/python3.11/site-packages/sklearn/multiclass.py:90

🚀 Vectorización de Características (TF-IDF y N-Grama)

In [ ]:
# --- Preparación de Variables para Multi-Etiqueta (Necesario para el split) ---
toxic_cols = ['IsToxic', 'IsAbusive', 'IsThreat', 'IsProvocative', 'IsObscene', 
              'IsHatespeech', 'IsRacist', 'IsNationalist', 'IsSexist', 
              'IsHomophobic', 'IsReligiousHate', 'IsRadicalism']

# --- 1. Separación de Datos (Train/Test Split) ---
# Definir las características (X) y la etiqueta principal (Y)
X = df['Text_Processed']
Y = df['IsToxic'] # Etiqueta principal para el split

# 80% Entrenamiento, 20% Prueba. Usamos 'stratify=Y' para mantener la proporción de clases.
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, 
    test_size=0.2, 
    random_state=42, 
    stratify=Y # Crucial para el desbalance
)

print(f"Datos separados: {len(X_train)} muestras de entrenamiento, {len(X_test)} de prueba.")

# También separamos las etiquetas multi-etiqueta para el entrenamiento posterior:
Y_multi_train = df.iloc[X_train.index][toxic_cols] 
Y_multi_test = df.iloc[X_test.index][toxic_cols]


# --- 2. Vectorización TF-IDF (Term Frequency-Inverse Document Frequency) ---

# CONVERSIÓN CRÍTICA: Convertimos el frozenset a list (Corregido el error)
stop_words_list = list(ENGLISH_STOP_WORDS) 

# Inicializar el Vectorizador TF-IDF con N-gramas y parámetros específicos
vectorizer = TfidfVectorizer(
    # Incluir unigramas y bigramas (N-gramas)
    ngram_range=(1, 2), 
    
    # Usamos la lista convertida para evitar el error 'InvalidParameterError'
    stop_words=stop_words_list,
    
    # Ignorar términos que aparecen en más del 90% de los documentos (demasiado comunes)
    max_df=0.9, 
    
    # Ignorar términos que aparecen en menos de 5 documentos (demasiado raros)
    min_df=5 
)

# A. Ajustar y Transformar SOLO en el conjunto de ENTRENAMIENTO
X_train_vectorized = vectorizer.fit_transform(X_train)

# B. Transformar el conjunto de PRUEBA (usando el vocabulario aprendido en el entrenamiento)
X_test_vectorized = vectorizer.transform(X_test)

print("\n--- Resultados de la Vectorización ---")
print(f"Dimensión de los datos de Entrenamiento: {X_train_vectorized.shape}")
print(f"Dimensión de los datos de Prueba: {X_test_vectorized.shape}")
print(f"Número total de características (N-gramas): {X_train_vectorized.shape[1]}")

Datos separados: 800 muestras de entrenamiento, 200 de prueba.

--- Resultados de la Vectorización ---
Dimensión de los datos de Entrenamiento: (800, 569)
Dimensión de los datos de Prueba: (200, 569)
Número total de características (N-gramas): 569
